In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%pip install ../databricks_helpers
%pip install meteostat
dbutils.library.restartPython()

In [0]:
from databricks_helpers.databricks_helper import DatabricksHelpers
dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

In [0]:
%sql
--CREATE VOLUME IF NOT EXISTS raw_data;
SELECT current_catalog(), current_schema();

In [0]:
#IATA_CODE to LAT, LONG taken from https://ourairports.com/data/
df=spark.read.csv(f"{dbh.volume_directory()}/airports.csv", header=True, inferSchema=True)


In [0]:
df.filter(df.iata_code.isNotNull())\
.withColumnsRenamed({"longitude_deg":"long","iata_code":"iata","latitude_deg":"lat"})\
.drop("id","iso_region","municipality","scheduled_service","gps_code","local_code","home_link","wikipedia_link","keywords")\
.write.mode("overwrite").saveAsTable("airports")

In [0]:
%sql
CREATE VIEW if not exists req_airports_view as select * From airports where iata in (select distinct ORIGIN from bronze_flight_data); -- required otherwise the airport count is more than 500

In [0]:
from datetime import datetime
import meteostat as ms
import os


meteo_daily_data=f"{dbh.volume_directory()}/daily_meteo_data"
os.makedirs(meteo_daily_data)


weather_list = []
required_locations = [(row.iata, (row.lat, row.long)) for row in spark.table("req_airports_view").select("iata","lat","long").collect()]

In [0]:
import time
import pickle
import os

monthly_limit = 500
requests_made = 0
batch_size = 3
checkpoint_file = f"{dbh.volume_directory()}/weather_checkpoint.pkl"
# Load checkpoint if exists, so that if any batch fails it can start from there
#checkpointing in case there is a failure
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
        start_idx = checkpoint.get("start_idx", 0)
        requests_made = checkpoint.get("requests_made", 0)
        weather_list = checkpoint.get("weather_list", [])
else:
    start_idx = 0

for i in range(start_idx, min(len(required_locations), monthly_limit), batch_size): #iteracting batch of 3 so as to abide the rate limit of 3 req/second
    batch = required_locations[i:i+batch_size]
    for code, (lat, lon) in batch:
        if requests_made >= monthly_limit:
            break
        point = ms.Point(lat, lon)
        data = ms.Daily(point, start, end).fetch().reset_index()
        data['iata_code'] = code
        weather_list.append(data)
        requests_made += 1
    # Save checkpoint after each batch
    with open(checkpoint_file, "wb") as f:
        pickle.dump({
            "start_idx": i + batch_size,
            "requests_made": requests_made,
            "weather_list": weather_list
        }, f)
    time.sleep(1)  # Enforce 3 requests per second

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import meteostat as ms
import pandas as pd
import time

spark = SparkSession.builder.getOrCreate()

monthly_limit = 500
batch_size = 3

# Convert input list to Spark DataFrame
df = spark.createDataFrame(required_locations, ["iata_code", "coords"])
#Split to lat long
df = df.withColumn("lat", col("coords._1")) \
       .withColumn("lon", col("coords._2")).drop("coords")


In [0]:

# Limit total requests
df = df.limit(monthly_limit)

# Repartition to control parallelism
df = df.repartition(2)  # tune this based on cluster size

def fetch_weather_partition(iterator):
    import meteostat as ms
    import pandas as pd
    import time

    results = []
    requests_made = 0

    batch = []

    for row in iterator:
        batch.append(row)

        if len(batch) == batch_size:
            for r in batch:
                try:
                    point = ms.Point(r.lat, r.lon)
                    station = ms.stations.nearby(point, limit=1)
                    data = ms.daily(station, start, end).fetch().reset_index()
                    data['iata_code'] = r.iata_code
                    results.append(data)
                    requests_made += 1
                except Exception as e:
                    print(f"Error: {e}")

            batch = []
            time.sleep(1)  # rate limit (3 req/sec)

    # leftover batch, if batch not multiple of 4
    for r in batch:
        try:
            point = ms.Point(r.lat, r.lon)
            station = ms.stations.nearby(point, limit=1)
            data = ms.daily(station, start, end).fetch().reset_index()
            data['iata_code'] = r.iata_code
            results.append(data)
        except Exception as e:
            print(f"Error: {e}")

    if results:
        return iter([pd.concat(results).to_dict(orient="records")])  # convert to list of dicts to avoid pandas internals
        #return iter(final_df.to_dict(orient="records")) 
    else:
        return iter([])

# Apply distributed processing
weather_rdd = df.rdd.mapPartitions(fetch_weather_partition)

# Convert back to Spark DataFrame
#weather_df = spark.createDataFrame(weather_rdd) doesnt work with pd.condat and returning pandas dataframe



In [0]:
point = ms.Point(39.2966, -80.228104)
station = ms.stations.nearby(point, limit=1)
# Fetch data
data = ms.daily(station, start, end).fetch().reset_index()
data

In [0]:
import pyspark.sql.functions as F
from datetime import datetime

output_schema = """
    time TIMESTAMP,
    temp DOUBLE,
    tmin DOUBLE,
    tmax DOUBLE,
    rhum DOUBLE,
    prcp DOUBLE,
    snwd DOUBLE,
    wspd DOUBLE,
    wpgt DOUBLE,
    pres DOUBLE,
    tsun DOUBLE,
    cldc DOUBLE,
    iata_code STRING
"""
meteo_dir=f"{dbh.volume_directory()}/meteo_data"
os.makedirs(meteo_dir, exist_ok=True)

batch_size = 4
years = [2022, 2023, 2024, 2025]
meteo_dir = f"{dbh.volume_directory()}/meteo_data"

# 🔥 Databricks-safe existence check
def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except:
        return False

# Find missing years
missing_years = []
for year in years:
    year_dir = f"{meteo_dir}/meteo_weather_data_{year}"
    if not path_exists(year_dir):
        missing_years.append(year)

# 🔥 Optimize: prepare df ONCE
df_prepared = df.limit(monthly_limit).repartition(2)

def fetch_weather_pandas(iterator):
    import meteostat as ms
    import pandas as pd
    import time

    expected_cols = [
        'time','temp','tmin','tmax','rhum','prcp',
        'snwd','wspd','wpgt','pres','tsun','cldc','iata_code'
    ]

    for pdf in iterator:
        results = []
        requests_made = 0

        for _, row in pdf.iterrows():
            try:
                point = ms.Point(row['lat'], row['lon'])
                station=ms.stations.nearby(point, limit=1)
                # ✅ use global start/end (set per loop)
                print(f"{start}/{end}")
                data = ms.daily(station, start, end).fetch().reset_index()

                if not data.empty:
                    data['iata_code'] = row['iata_code']
                    data = data.reindex(columns=expected_cols)
                    results.append(data)

                requests_made += 1

                if requests_made % batch_size == 0:
                    time.sleep(1)

            except Exception as e:
                print(f"Error fetching data for {row.get('iata_code', 'Unknown')}: {e}")

        if results:
            final_pdf = pd.concat(results, ignore_index=True)
            # 🔥 This removed chunked array issue of arrow, below code forces pd to use numpy array
            final_pdf = pd.DataFrame({
                col: final_pdf[col].to_numpy()
                for col in final_pdf.columns
            })
            yield final_pdf
        else:
            empty_df = pd.DataFrame(columns=expected_cols)
            empty_df = pd.DataFrame({
                col: empty_df[col].to_numpy()
                for col in empty_df.columns
            })
            yield empty_df

In [0]:
# 🚀 MAIN LOOP (only change you needed)
for year in missing_years:
    print(f"Processing year: {year}")

    start = datetime(year, 1, 1)
    end = datetime(year, 12, 31)# by pythons late binding it works

    weather_df = df_prepared.mapInPandas(fetch_weather_pandas, schema=output_schema)

    output_path = f"{meteo_dir}/meteo_weather_data_{year}"
    weather_df.write.mode("overwrite").json(output_path)

    print(f"Saved: {output_path}")

In [0]:
display(weather_df)